<span style="color:lightgreen">

# L4.2 Transcripción con Traducción del portugués al inglés EuroparlST

<span>

# ST Baseline experiment using Whisper and Europarl-ST (Spanish-English)


In this notebook, we are going to learn how to use the Open AI pre-trained model [Whisper](https://openai.com/index/whisper/) for speech translation on the [Europarl-ST dataset](https://huggingface.co/datasets/tj-solergibert/Europarl-ST) (using Spanish-English speech data).

First, we import some OpenAI source whisper libraries and additional ones (e.g. for computing evaluation figures: WER and BLEU)

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())


%load_ext autoreload
%autoreload 2
%matplotlib inline

from src.config import Configuration

CONFIG = Configuration()

/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/notebooks_pt_en
/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/app


In [2]:
import whisper
import jiwer
from whisper.normalizers import BasicTextNormalizer

from tqdm.notebook import tqdm
import pandas as pd

model = whisper.load_model("base")

Load Europarl-ST Spanish-English test audio dataset

In [3]:
from datasets import load_dataset

raw_datasets = load_dataset("facebook/covost2", CONFIG.trans_code, data_dir=CONFIG.corpus_path)

print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['client_id', 'file', 'audio', 'sentence', 'translation', 'id'],
        num_rows: 9158
    })
    validation: Dataset({
        features: ['client_id', 'file', 'audio', 'sentence', 'translation', 'id'],
        num_rows: 3318
    })
    test: Dataset({
        features: ['client_id', 'file', 'audio', 'sentence', 'translation', 'id'],
        num_rows: 4023
    })
})


<p style="page-break-after:always;"></p>

Translate to English all the audio data using the Whisper (base) model. Automatic translations are stored in translations. At the same time, translation references and source transcriptions are stored into references and sources, respectively.

In [4]:
data = raw_datasets["test"]
translations = []

for sample in tqdm(data["file"]):   
    translations.append((model.transcribe(sample, language=CONFIG.src_name, task="translate"))['text'])


  0%|          | 0/4023 [00:00<?, ?it/s]

<p style="page-break-after:always;"></p>

Automatic translations, references and sources are stored into a Pandas dataframe. Show the two first translations and references.

In [5]:
data_df = pd.DataFrame(dict(translation=translations, reference=data["translation"], source=data["sentence"]))
pd.set_option('display.max_colwidth', None)
data_df.head(2)

,translation,reference,source
0,day of money and press the people of the yoday Mun seguir,Borrow money from people in the village,Pedir dinheiro emprestado às pessoas da aldeia
1,Drもう靠,Lock them up,Trancá-los


Automatic translations, references and sources are normalized using the Whisper basic text standardisation/normalization module

In [6]:
normalizer = BasicTextNormalizer()

data_df["translation_clean"] = [normalizer(text) for text in data_df["translation"]]
data_df["reference_clean"] = [normalizer(text) for text in data_df["reference"]]
data_df["source_clean"] = [normalizer(text) for text in data_df["source"]]
data_df.head(2)

,translation,reference,source,translation_clean,reference_clean,source_clean
0,day of money and press the people of the yoday Mun seguir,Borrow money from people in the village,Pedir dinheiro emprestado às pessoas da aldeia,day of money and press the people of the yoday mun seguir,borrow money from people in the village,pedir dinheiro emprestado às pessoas da aldeia
1,Drもう靠,Lock them up,Trancá-los,drもう靠,lock them up,trancá los


<p style="page-break-after:always;"></p>

For evaluation, we use the [Evaluate library](https://huggingface.co/docs/evaluate) which includes the definition of generic and task-specific metrics. In our case, we use the [BLEU metric](https://huggingface.co/spaces/evaluate-metric/bleu), or to be more precise, [sacreBLEU](https://huggingface.co/spaces/evaluate-metric/sacrebleu).

In [7]:
from evaluate import load

metric = load("sacrebleu")

In [8]:
result = metric.compute(predictions=data_df["translation_clean"], references=data_df["reference_clean"])
print(f'BLEU score: {result["score"]:.1f}')

BLEU score: 22.8


Compute COMET figures using the [Evaluate library](https://huggingface.co/docs/evaluate) which includes the definition of generic and task-specific metrics.

In [9]:
from evaluate import load
comet_metric = load('comet')

/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/venv/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/venv/lib/python3.12/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [10]:
comet_score = comet_metric.compute(predictions=data_df["translation_clean"], references=data_df["reference_clean"], sources=data_df["source_clean"])

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 5070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


In [11]:
print(f"COMET: {comet_score['mean_score'] * 100:.2f} %")

COMET: 62.55 %


All the data is stored into a file using 'csv' format

In [12]:
data_df.to_csv(os.path.join(CONFIG.RESULTS_PATH, 'L4.2_ST_Whisper_Baseline_Europarl-ST.csv'), encoding='utf-8')

# Exercise

Perform a similar experiment using the Covost2 source-english setup previously used in L4.1. Evaluate the performance of different whisper models 